# Transformer Block 완전 구현 - 실습 코드 1: 완전한 Transformer Encoder
### (상세 설명판 - 처음 배우는 분들을 위해 개념 설명과 예제를 보강한 버전)

- Tutorial ID: `expand-transformer-block-impl`
- Tutorial: Transformer Block 완전 구현
- Section ID: `expand-transformer-block-impl-code-1`
- Section: 실습 코드 1: 완전한 Transformer Encoder

---

이 노트북은 원본 실습 코드에 **개념 설명, 작은 숫자로 직접 계산해보는 예제, 각 단계별 실제 출력**을 더한 버전입니다.
새 용어가 나오기 전에는 반드시 먼저 말로 설명하고, 이어서 작은 예제로 직접 눈으로 확인한 뒤, 마지막에 실제 구현 코드를 보는 순서로 구성했습니다.

## 이 노트북을 보는 법

이 노트북은 "정답 코드를 한 번 실행해보는 용도"가 아니라, **수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을 한 줄씩 추적**하기 위한 실습 노트입니다.

### 학습 목표
1. Q(Query)/K(Key)/V(Value)가 어떤 shape으로 만들어지고, 그 값들이 어떻게 attention score로 이어지는지 직접 손으로 추적합니다.
2. 미래 토큰을 `-inf`로 가린 뒤, softmax를 거치면 그 확률이 정확히 0이 되는 과정을 숫자로 직접 확인합니다.

### 노트북 구성
이 노트북은 크게 두 파트로 나뉩니다.

| 구분 | 내용 | 필요한 도구 | 이 환경에서 바로 실행? |
|---|---|---|---|
| **Part A** | NumPy로 원리를 손으로 계산해보기 (Embedding·Attention·Mask·FFN·LayerNorm) | NumPy만 있으면 됨 | ✅ 가능 (아래 코드가 이미 실행된 결과도 함께 보입니다) |
| **Part B** | 같은 내용을 실제 PyTorch `nn.Module`로 구현하기 (원본 실습 코드 + 상세 주석) | PyTorch 설치 필요 | Colab 또는 로컬 환경에서 실행 |

즉, **Part A에서 "왜 이 연산을 하는지"를 숫자로 먼저 확인**하고 → **Part B에서 "PyTorch가 그것을 얼마나 짧게 자동화해주는지"**를 확인하는 순서입니다. 새로운 개념은 모두 Part A에서 먼저 설명되므로, Part B의 코드는 Part A를 떠올리며 "아, 그게 이렇게 구현되는구나" 하고 읽으시면 됩니다.

### 읽는 순서 (권장)
1. 차원/하이퍼파라미터(`batch_size`, `seq_len`, `d_model` 등)를 먼저 확인합니다.
2. 입력 배열 `x` 또는 토큰 데이터가 어떻게 만들어지는지 봅니다.
3. `W_Q`/`W_K`/`W_V`/`W_O` 같은 가중치 행렬이 어떤 공간으로 값을 투영하는지 확인합니다.
4. `@`(행렬곱), `softmax`, `mask` 같은 핵심 연산 직후의 shape과 값을 출력으로 직접 검증합니다.
5. `seed`, 차원, `num_heads` 같은 값을 바꿔가며 결과가 어떻게 달라지는지 실험해봅니다 (Part B 마지막 "직접 실험해보기" 참고).

### 주의
- 숫자 하나하나를 외우기보다 **"shape이 어떻게 변하는지"**와 **"정보가 어느 방향으로 흘러가는지"**를 눈으로 따라가 보세요.
- Part B의 PyTorch 코드는 이 실습 환경에 PyTorch가 설치되어 있지 않아 셀 출력이 비어 있습니다. Google Colab(기본 설치되어 있음) 또는 PyTorch가 설치된 로컬/서버 환경에 그대로 복사해서 실행해보세요. (설치 방법은 Part B 시작 부분에 안내되어 있습니다)

## 전체 그림: Transformer Encoder는 어떤 모양일까?

코드를 보기 전에, 우리가 최종적으로 만들 모델의 전체 데이터 흐름을 먼저 눈에 익혀둡시다.
아래는 Encoder 한 겹(layer), 즉 `EncoderBlock` 하나의 흐름입니다. 이 블록을 여러 겹 쌓으면 전체 Transformer Encoder가 됩니다.

```
토큰 ID들 (입력 문장)
        |
        v
Token Embedding + Positional Encoding      <- Part A 1, 2
        |
        v
        x
        |
        +------------------+
        v                  |
    LayerNorm              |             <- Part A 7
        v                  |
  Self-Attention           |             <- Part A 3, 4
        v                  |
       (+) <---------------+     residual connection #1
        |
        +------------------+
        v                  |
    LayerNorm              |             <- Part A 7
        v                  |
   FeedForward             |             <- Part A 6
        v                  |
       (+) <---------------+     residual connection #2
        |
        v
   EncoderBlock 출력  (다음 EncoderBlock의 입력으로, N번 반복)
```

마지막 `EncoderBlock`을 통과한 뒤에는 LayerNorm을 한 번 더 적용해 최종 출력을 만듭니다. 이 다이어그램에 나오는 구성 요소 하나하나를 Part A에서 작은 숫자로 직접 계산해보겠습니다.

---
# Part A. 원리부터 이해하기 (NumPy)

이 파트의 코드는 PyTorch가 전혀 필요 없는 **순수 NumPy**로 작성되어 있어서, 이 실습 환경에서 바로 실행한 결과를 아래에서 확인할 수 있습니다.
아주 작은 숫자(단어 3개, 4차원 벡터)로 계산하기 때문에 출력된 값을 하나하나 눈으로 따라갈 수 있습니다.

앞으로 계속 사용할 예문은 다음과 같습니다.

> **"나는 밥을 먹었다"**

이 세 단어가 Embedding → Positional Encoding → Self-Attention → (Mask) → FeedForward → LayerNorm 을 거치며 어떻게 모양이 바뀌고, 어떤 값으로 채워지는지 끝까지 따라가 봅시다.

In [1]:
import numpy as np

# 실행할 때마다 같은 난수가 나오도록 시드를 고정합니다.
# (재현성을 위해서입니다 - 시드를 고정하지 않으면 실행할 때마다 값이 달라져서 아래 설명과 비교하기 어렵습니다)
np.random.seed(42)

print("NumPy version:", np.__version__)
print("이 시드로 앞으로 나오는 모든 '무작위' 값이 항상 같은 값으로 재현됩니다.")

NumPy version: 2.4.4
이 시드로 앞으로 나오는 모든 '무작위' 값이 항상 같은 값으로 재현됩니다.

## 1. 토큰(token)과 임베딩(embedding)이란?

컴퓨터는 `"나는"`, `"밥을"` 같은 글자를 그대로 이해하지 못합니다. 숫자만 계산할 수 있기 때문입니다.

그래서 딥러닝 모델은 두 단계를 거쳐 글자를 숫자로 바꿉니다.

1. **토큰화(tokenization)** : 각 단어(또는 더 작은 단위)에 고유한 번호(ID)를 붙입니다. 예: `"나는"` → `0`, `"밥을"` → `1`
2. **임베딩(embedding)** : 그 번호를 '의미를 담을 수 있는' 벡터(숫자 여러 개로 이루어진 목록)로 바꿉니다.

임베딩은 사실 아주 단순합니다. **(단어 개수) x (벡터 차원)** 크기의 표를 하나 만들어두고, 필요한 단어의 번호에 해당하는 행(row)을 그대로 꺼내오는 것이 전부입니다. 이 표는 처음엔 무작위 값으로 시작하지만, 학습을 거치면서 점점 '의미 있는' 값으로 바뀝니다 (예: "고양이"와 "강아지"의 벡터가 서로 가까워지는 식으로).

아래 코드에서 5개 단어짜리 아주 작은 사전으로 직접 확인해봅니다.

In [2]:
# ------------------------------------------------------------
# 아주 작은 '미니 사전' (실제로는 수만~수십만 단어가 들어있습니다)
# ------------------------------------------------------------
vocab = {"나는": 0, "밥을": 1, "먹었다": 2, "고양이": 3, "달린다": 4}
vocab_size = len(vocab)   # 5
d_model = 4                # 각 단어를 4개의 숫자(4차원 벡터)로 표현합니다 (실제 모델은 보통 256~4096차원)

# '임베딩 테이블' = (단어 개수 x 벡터 차원) 크기의 표.
# 아직 학습 전이므로 완전히 무작위인 값입니다 - 숫자 자체의 의미보다 "구조"에 집중하세요.
embedding_table = np.random.randn(vocab_size, d_model) * 0.1
print("임베딩 테이블 shape:", embedding_table.shape, " (단어 5개 x 4차원)")
print(np.round(embedding_table, 3))

# "나는 밥을 먹었다" 라는 문장을 토큰 ID로 바꾸면 [0, 1, 2] 입니다.
sentence = ["나는", "밥을", "먹었다"]
token_ids = np.array([vocab[w] for w in sentence])
print("\n문장:", sentence)
print("토큰 ID:", token_ids)

# 임베딩 테이블에서 각 ID에 해당하는 '행'을 그대로 꺼내오는 것 -> 이것이 임베딩의 전부입니다.
token_emb = embedding_table[token_ids]
print("\n토큰 임베딩 shape:", token_emb.shape, " (토큰 3개 x 4차원)")
print(np.round(token_emb, 3))

임베딩 테이블 shape: (5, 4)  (단어 5개 x 4차원)
[[ 0.05  -0.014  0.065  0.152]
 [-0.023 -0.023  0.158  0.077]
 [-0.047  0.054 -0.046 -0.047]
 [ 0.024 -0.191 -0.172 -0.056]
 [-0.101  0.031 -0.091 -0.141]]

문장: ['나는', '밥을', '먹었다']
토큰 ID: [0 1 2]

토큰 임베딩 shape: (3, 4)  (토큰 3개 x 4차원)
[[ 0.05  -0.014  0.065  0.152]
 [-0.023 -0.023  0.158  0.077]
 [-0.047  0.054 -0.046 -0.047]]

## 2. 왜 위치 정보(Positional Encoding)가 필요할까?

뒤에서 만들 Self-Attention은 "이 단어가 몇 번째에 있는지"를 스스로 구분하지 못합니다. Self-Attention만 놓고 보면 "나는 밥을 먹었다"와 "먹었다 밥을 나는"의 단어들 사이 관계 계산 결과가 사실상 동일합니다 (단어들을 순서 없이 뒤섞인 '가방(bag)'처럼 다루기 때문입니다).

그래서 "몇 번째 위치인지"를 알려주는 벡터를 따로 만들어 단어 임베딩에 더해줍니다. 이를 위치 인코딩(Positional Encoding)이라고 부릅니다.

> 참고: 위치 인코딩을 만드는 방법은 크게 두 가지입니다.
> - **고정 수식(sinusoidal)** : 원 논문(Attention is All You Need)에서 사용한 방식으로, sin/cos 함수로 위치마다 고유한 패턴을 계산합니다. 학습되지 않고 고정되어 있습니다.
> - **학습형(learned)** : 단어 임베딩처럼, "위치"도 하나의 조회 테이블로 만들어 학습 과정에서 값이 바뀌도록 합니다.
>
> 이 노트북(과 Part B의 실제 구현)에서는 구현이 더 간단한 **학습형**을 사용합니다. 즉, 단어 임베딩 테이블을 하나 더 만드는 것과 완전히 같은 방식이며, 다만 "단어 종류"가 아니라 "위치(0번째, 1번째, ...)"마다 벡터가 하나씩 있다는 점만 다릅니다.

In [3]:
max_seq_len = 10  # 문장이 최대 몇 단어까지 올 수 있는지 (이보다 긴 문장은 이 표로 처리할 수 없습니다)

# 위치 임베딩 테이블도 단어 임베딩과 완전히 같은 구조입니다.
position_table = np.random.randn(max_seq_len, d_model) * 0.1
print("위치 임베딩 테이블 shape:", position_table.shape, " (최대 위치 10개 x 4차원)")

positions = np.arange(len(sentence))  # [0, 1, 2] -> "나는"=0번째, "밥을"=1번째, "먹었다"=2번째
pos_emb = position_table[positions]
print("\n이번 문장(3단어)에 쓰인 위치 임베딩 shape:", pos_emb.shape)
print(np.round(pos_emb, 3))

# 최종 입력 벡터 = 단어의 '의미' 정보 + '위치' 정보
x = token_emb + pos_emb
print("\n최종 입력 x shape:", x.shape, " (단어 의미 + 위치 정보가 더해짐, shape은 그대로 유지)")
print(np.round(x, 3))

위치 임베딩 테이블 shape: (10, 4)  (최대 위치 10개 x 4차원)

이번 문장(3단어)에 쓰인 위치 임베딩 shape: (3, 4)
[[ 0.147 -0.023  0.007 -0.142]
 [-0.054  0.011 -0.115  0.038]
 [-0.06  -0.029 -0.06   0.185]]

최종 입력 x shape: (3, 4)  (단어 의미 + 위치 정보가 더해짐, shape은 그대로 유지)
[[ 0.196 -0.036  0.072  0.01 ]
 [-0.078 -0.012  0.043  0.114]
 [-0.107  0.025 -0.107  0.139]]

## 3. Self-Attention이란? — Query, Key, Value

비유를 하나 들어보겠습니다. 유튜브에서 영상을 검색하는 상황을 떠올려보세요.

- **Query (질의)** : 내가 검색창에 입력하는 검색어 → "내가 지금 찾고 있는 것"
- **Key (열쇠)** : 각 영상에 달려있는 제목/태그 → "각 토큰이 내세우는 자기 자신의 특징표"
- **Value (값)** : 영상이 실제로 담고 있는 내용 → "각 토큰이 실제로 전달할 정보"

검색어(Query)와 각 영상의 태그(Key)가 얼마나 잘 맞는지 점수를 매기고, 그 점수(=관련도)를 가중치 삼아 각 영상의 실제 내용(Value)을 섞으면 "내가 찾던 것에 가까운 결과"가 만들어집니다. Self-Attention은 문장 속 단어들끼리 정확히 이 과정을 수행합니다: 각 단어가 Query가 되어 문장 속 모든 단어(자기 자신 포함)의 Key와 관련도를 계산하고, 그 관련도로 Value들을 섞어 자신의 새로운 표현을 만듭니다.

공식으로 쓰면 다음과 같습니다.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

이 공식을 4단계로 나누어 손으로 직접 계산해보겠습니다.

1. `Q @ K.T` : 모든 단어 쌍의 유사도 점수 계산
2. `/ sqrt(d_k)` : 점수가 너무 커지지 않도록 스케일 조정
3. `softmax(...)` : 점수를 "합이 1인 확률(가중치)"로 변환
4. `... @ V` : 그 가중치로 Value들을 가중 평균 → 최종 출력

In [4]:
d_k = d_model  # 이번 예시에서는 Q, K, V의 차원을 d_model과 동일하게 사용합니다.

# 실제 모델에서는 W_Q, W_K, W_V라는 '학습되는 가중치 행렬'을 x에 곱해서 Q, K, V를 만듭니다.
# 아직 학습 전이므로 지금은 무작위 값으로 그 역할만 흉내내 보겠습니다.
W_q = np.random.randn(d_model, d_k) * 0.5
W_k = np.random.randn(d_model, d_k) * 0.5
W_v = np.random.randn(d_model, d_k) * 0.5

Q = x @ W_q   # "각 토큰이 무엇을 찾고 있는가"
K = x @ W_k   # "각 토큰이 내세우는 특징"
V = x @ W_v   # "각 토큰이 실제로 담고 있는 정보"

print("Q shape:", Q.shape, " (토큰 3개, 각 4차원)")
print("K shape:", K.shape)
print("V shape:", V.shape)

# [1단계] Q와 K를 곱해 '모든 토큰 쌍'의 유사도 점수를 한 번에 계산합니다.
# (3,4) @ (4,3) -> (3,3)  ->  [i, j] = i번째 토큰의 Query와 j번째 토큰의 Key가 얼마나 비슷한가
scores = Q @ K.T
print("\n[1단계] 유사도 점수 Q·K^T, shape:", scores.shape)
print(np.round(scores, 3))

# [2단계] 스케일링: d_k가 커질수록 점수의 절댓값이 커지는 경향이 있어,
# 그대로 softmax에 넣으면 한쪽 값만 극단적으로 커져 학습이 잘 안 됩니다. 그래서 sqrt(d_k)로 나눠줍니다.
scaled_scores = scores / np.sqrt(d_k)
print("\n[2단계] sqrt(d_k)=%.2f 로 나눈 뒤:" % np.sqrt(d_k))
print(np.round(scaled_scores, 3))

# [3단계] softmax로 '각 행의 합이 1인 확률'로 바꿔줍니다.
def softmax(z):
    z = z - np.max(z, axis=-1, keepdims=True)  # 오버플로 방지용 안전장치 (결과는 수학적으로 동일)
    e = np.exp(z)
    return e / np.sum(e, axis=-1, keepdims=True)

attn_weights = softmax(scaled_scores)
print("\n[3단계] Softmax 이후 Attention 가중치, shape:", attn_weights.shape)
print(np.round(attn_weights, 3))
print("각 행의 합 (1이어야 정상):", attn_weights.sum(axis=-1))

# [4단계] 가중치로 V들을 '가중 평균'하면 각 토큰의 새로운 표현이 완성됩니다.
attn_output = attn_weights @ V
print("\n[4단계] 최종 Attention 출력, shape:", attn_output.shape, " (입력 x와 shape이 동일!)")
print(np.round(attn_output, 3))

Q shape: (3, 4)  (토큰 3개, 각 4차원)
K shape: (3, 4)
V shape: (3, 4)

[1단계] 유사도 점수 Q·K^T, shape: (3, 3)
[[ 0.015 -0.011 -0.013]
 [-0.026  0.017  0.027]
 [-0.02   0.01   0.026]]

[2단계] sqrt(d_k)=2.00 로 나눈 뒤:
[[ 0.008 -0.006 -0.007]
 [-0.013  0.009  0.013]
 [-0.01   0.005  0.013]]

[3단계] Softmax 이후 Attention 가중치, shape: (3, 3)
[[0.336 0.332 0.332]
 [0.328 0.335 0.337]
 [0.329 0.334 0.337]]
각 행의 합 (1이어야 정상): [1. 1. 1.]

[4단계] 최종 Attention 출력, shape: (3, 4)  (입력 x와 shape이 동일!)
[[-0.012  0.015  0.081  0.004]
 [-0.01   0.016  0.082  0.006]
 [-0.011  0.016  0.082  0.006]]

## 4. Multi-Head Attention: 왜 여러 개의 head가 필요할까?

방금 계산한 것은 head가 1개인 경우입니다. 하지만 실제 Transformer는 `d_model` 차원을 여러 조각(head)으로 나누어 **동시에 여러 관점**에서 attention을 계산합니다. (예를 들어 어떤 head는 문법적 관계에, 다른 head는 의미적 관계에 자연스럽게 더 집중하게 되기도 합니다.)

나누는 방법은 간단합니다: `d_model` 차원을 `num_heads` 개로 **균등하게** 쪼갭니다.

$$d_{head} = d_{model} / \text{num\_heads}$$

그래서 **`d_model`은 반드시 `num_heads`로 나누어떨어져야 합니다.** 이것은 처음 Transformer를 구현할 때 아주 흔히 만나는 실수이니 꼭 기억해두세요.

In [5]:
num_heads = 2
assert d_model % num_heads == 0, "d_model은 num_heads로 나누어떨어져야 합니다!"
d_head = d_model // num_heads
print(f"d_model={d_model} 을 num_heads={num_heads} 개로 나누면, head 하나당 차원 d_head={d_head}")

# 방금 만든 Q (토큰 3개 x 4차원)를 head 2개로 나눠보겠습니다.
print("\n나누기 전 Q shape:", Q.shape)

Q_heads = Q.reshape(len(sentence), num_heads, d_head)
print("reshape 후 Q shape:", Q_heads.shape, " (토큰 수, head 수, head별 차원)")

# 이후 계산 순서에 맞게 축 순서를 바꾸면 (head 수, 토큰 수, head별 차원) 형태가 되고,
# 이 상태에서 head마다 '독립적으로' 위에서 했던 attention 계산을 수행한 뒤 결과를 다시 이어붙입니다(concat).
Q_heads = Q_heads.transpose(1, 0, 2)
print("transpose 후 Q shape:", Q_heads.shape, " -> head별로 독립적인 (3,2) 행렬이 2개 생긴 것")

# d_model이 num_heads로 나누어떨어지지 않으면 어떻게 될까요?
# 실제로 자주 만나게 되는 에러 상황을 미리 확인해봅니다.
try:
    bad_num_heads = 3
    assert d_model % bad_num_heads == 0, (
        f"d_model({d_model})이 num_heads({bad_num_heads})로 나누어떨어지지 않습니다!"
    )
except AssertionError as e:
    print("\n[예상되는 에러 상황]", e)

d_model=4 을 num_heads=2 개로 나누면, head 하나당 차원 d_head=2

나누기 전 Q shape: (3, 4)
reshape 후 Q shape: (3, 2, 2)  (토큰 수, head 수, head별 차원)
transpose 후 Q shape: (2, 3, 2)  -> head별로 독립적인 (3,2) 행렬이 2개 생긴 것

[예상되는 에러 상황] d_model(4)이 num_heads(3)로 나누어떨어지지 않습니다!

## 5. 미래를 가리는 마스크(Causal Mask) — 왜 `-inf`를 사용할까?

> 참고: 지금 우리가 만드는 것은 Encoder(BERT 계열)이고, Encoder는 문장 전체를 양방향으로 봅니다. 즉 **Encoder 자체는 이 마스크가 필요 없습니다.**
> 하지만 GPT 같은 Decoder 모델은 "다음 단어 맞히기" 방식으로 학습하기 때문에, 학습 중 정답(미래 단어)을 미리 보면 반칙이 되어 미래 토큰을 반드시 가려야 합니다. 마스킹의 원리 자체는 Attention이 쓰이는 곳이라면 어디서든 똑같이 동작하므로, 여기서 실험을 통해 미리 확인해둡니다.

방법은 간단합니다. attention score를 계산한 뒤, "가려야 할 위치"에 `-inf`(음의 무한대)를 넣고 softmax를 적용합니다. softmax는 내부적으로 지수함수(`exp`)를 사용하는데, `exp(-inf) = 0` 이기 때문에 그 위치의 확률이 **정확히 0**이 되어 완전히 무시됩니다.

In [6]:
seq_len = len(sentence)  # 3   ("나는"=0, "밥을"=1, "먹었다"=2)

# 위쪽 삼각형 부분(자기 자신보다 미래인 위치)만 True로 표시합니다.
# [i, j] = True  <=>  j가 i보다 미래 위치 (j > i)
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)
print("Causal Mask (True = 가려야 할 '미래' 위치):")
print(causal_mask)
print("\n0행('나는')  : 자기 자신만 보고, '밥을'/'먹었다'는 아직 모르는 미래 취급")
print("2행('먹었다'): 앞의 모든 단어를 다 볼 수 있음 (마스킹 대상 없음)")

print("\n마스킹 전 점수 (Part A-3의 scaled_scores 재사용):")
print(np.round(scaled_scores, 3))

masked_scores = np.where(causal_mask, -np.inf, scaled_scores)
print("\n마스킹 후 점수 (미래 위치가 -inf로 바뀜):")
print(masked_scores)

masked_probs = softmax(masked_scores)
print("\nSoftmax 적용 후 확률 (미래 위치가 정확히 0.000이 되는지 확인!):")
print(np.round(masked_probs, 3))
print("\n각 행의 합 (마스킹 후에도 여전히 1이어야 정상):", masked_probs.sum(axis=-1))

# 확인: 마스킹의 영향을 전혀 받지 않는 마지막 행("먹었다")은
# Part A-3에서 마스킹 없이 계산했던 attn_weights의 마지막 행과 정확히 같아야 합니다.
print("\n[검증] masked_probs의 3번째 행 :", np.round(masked_probs[2], 3))
print("[검증] attn_weights의 3번째 행  :", np.round(attn_weights[2], 3))
print("[검증] 두 값이 같은가?          :", np.allclose(masked_probs[2], attn_weights[2]))

Causal Mask (True = 가려야 할 '미래' 위치):
[[False  True  True]
 [False False  True]
 [False False False]]

0행('나는')  : 자기 자신만 보고, '밥을'/'먹었다'는 아직 모르는 미래 취급
2행('먹었다'): 앞의 모든 단어를 다 볼 수 있음 (마스킹 대상 없음)

마스킹 전 점수 (Part A-3의 scaled_scores 재사용):
[[ 0.008 -0.006 -0.007]
 [-0.013  0.009  0.013]
 [-0.01   0.005  0.013]]

마스킹 후 점수 (미래 위치가 -inf로 바뀜):
[[ 0.00761323        -inf        -inf]
 [-0.01301535  0.00854869        -inf]
 [-0.00994386  0.00483399  0.01290516]]

Softmax 적용 후 확률 (미래 위치가 정확히 0.000이 되는지 확인!):
[[1.    0.    0.   ]
 [0.495 0.505 0.   ]
 [0.329 0.334 0.337]]

각 행의 합 (마스킹 후에도 여전히 1이어야 정상): [1. 1. 1.]

[검증] masked_probs의 3번째 행 : [0.329 0.334 0.337]
[검증] attn_weights의 3번째 행  : [0.329 0.334 0.337]
[검증] 두 값이 같은가?          : True

## 6. Feed-Forward Network(FFN)란? — 위치별로 독립적으로 적용되는 MLP

Self-Attention이 "토큰들 사이의 관계"를 봤다면, FFN은 각 토큰을 **다른 토큰과 상관없이 독립적으로** 한 번 더 가공합니다.

비유하자면: Attention에서 "주변 사람들 의견을 참고해서 내 생각을 업데이트"했다면, FFN은 "업데이트된 내 생각을 혼자 조용히 정리하는 단계"라고 볼 수 있습니다.

구조는 다음과 같습니다.

$$d_{model} \;\xrightarrow{\text{확장}}\; d_{ff} \;\xrightarrow{\text{활성화 함수}}\; d_{ff} \;\xrightarrow{\text{축소}}\; d_{model}$$

보통 `d_ff`는 `d_model`의 4배 정도로 설정합니다 (예: `d_model=512`이면 `d_ff=2048`). 차원을 넓혔다가 다시 좁히는 이유는, 중간에 더 넓은 공간을 거치게 하면 더 복잡한 패턴을 표현할 수 있는 여유가 생기기 때문입니다.

In [7]:
d_ff = 16  # 이번 예시에서는 d_model(4)의 4배로 설정합니다.

W1 = np.random.randn(d_model, d_ff) * 0.3
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d_model) * 0.3
b2 = np.zeros(d_model)

def gelu(z):
    # GELU는 ReLU를 부드럽게 만든 활성화 함수입니다. 실제 논문에서 쓰이는 근사식을 그대로 사용합니다.
    return 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z ** 3)))

# attn_output(Part A-3의 최종 attention 결과)을 입력으로 사용합니다.
hidden = gelu(attn_output @ W1 + b1)
print("1단계 확장 - hidden shape:", hidden.shape, f" (d_model={d_model} -> d_ff={d_ff})")

ffn_output = hidden @ W2 + b2
print("2단계 축소 - ffn_output shape:", ffn_output.shape, f" (d_ff={d_ff} -> 다시 d_model={d_model})")

print("\n입력과 출력의 shape이 같습니다:", attn_output.shape, "->", ffn_output.shape)
print(np.round(ffn_output, 3))

1단계 확장 - hidden shape: (3, 16)  (d_model=4 -> d_ff=16)
2단계 축소 - ffn_output shape: (3, 4)  (d_ff=16 -> 다시 d_model=4)

입력과 출력의 shape이 같습니다: (3, 4) -> (3, 4)
[[-0.013  0.025 -0.009  0.024]
 [-0.013  0.025 -0.009  0.025]
 [-0.013  0.025 -0.009  0.025]]

## 7. LayerNorm과 잔차 연결(Residual Connection) — 왜 필요할까?

**LayerNorm (층 정규화)**
각 토큰의 벡터를 "평균 0, 표준편차 1"이 되도록 정규화합니다. 배치 전체를 기준으로 정규화하는 BatchNorm과 달리, LayerNorm은 **토큰 하나하나**를 기준으로 정규화합니다. 그래서 문장 길이가 배치마다 달라도 안정적으로 동작하고, 학습을 훨씬 안정적으로 만들어줍니다.

**잔차 연결 (Residual Connection)**
서브층(Attention, FFN)을 통과하기 **전** 값을 다시 더해주는 것입니다.

$$\text{출력} = x + \text{서브층}(x)$$

층을 깊게 쌓아도 처음 정보가 통째로 사라지지 않도록 '지름길'을 만들어주는 역할을 합니다. 학습 시 기울기(gradient)가 역전파될 때도 이 지름길을 타고 흐를 수 있어서, 층이 깊어져도(예: 수십~수백 겹) 학습이 훨씬 안정적으로 이루어집니다.

In [8]:
def layer_norm(z, eps=1e-5):
    mean = z.mean(axis=-1, keepdims=True)
    var = z.var(axis=-1, keepdims=True)
    return (z - mean) / np.sqrt(var + eps)

# 일부러 평균/분산이 큰 값을 만들어 정규화 효과를 눈으로 확인해봅니다.
messy = attn_output * 50 + 100
print("정규화 전 - 전체 평균: %.2f, 전체 표준편차: %.2f" % (messy.mean(), messy.std()))

normed = layer_norm(messy)
print("정규화 후 - 전체 평균: %.2f, 전체 표준편차: %.2f" % (normed.mean(), normed.std()))
print("(각 '행'별로 평균 0, 표준편차 1이 되도록 계산되었기 때문에, 전체적으로 봐도 평균은 0, 표준편차는 1에 가깝습니다)")

# 잔차 연결 예시: FFN을 통과하기 전의 attn_output을 FFN 결과에 다시 더해줍니다.
residual_output = attn_output + ffn_output
print("\n잔차 연결 결과 shape:", residual_output.shape, " (attn_output + ffn_output, shape 변화 없음)")
print(np.round(residual_output, 3))

정규화 전 - 전체 평균: 101.15, 전체 표준편차: 1.76
정규화 후 - 전체 평균: -0.00, 전체 표준편차: 1.00
(각 '행'별로 평균 0, 표준편차 1이 되도록 계산되었기 때문에, 전체적으로 봐도 평균은 0, 표준편차는 1에 가깝습니다)

잔차 연결 결과 shape: (3, 4)  (attn_output + ffn_output, shape 변화 없음)
[[-0.025  0.04   0.072  0.028]
 [-0.024  0.041  0.073  0.031]
 [-0.024  0.041  0.073  0.031]]

## Part A 정리

지금까지 NumPy로 손수 계산해본 내용을 정리하면 다음과 같습니다.

| 단계 | Part A에서 우리가 한 일 | PyTorch에서는 |
|---|---|---|
| 임베딩 | `embedding_table[token_ids]` | `nn.Embedding` |
| 위치 인코딩 | `position_table[positions]` | `nn.Embedding` (하나 더) |
| Self-Attention | `softmax(Q @ K.T / sqrt(d_k)) @ V` 직접 계산 | `nn.MultiheadAttention` |
| 마스킹 | `np.where(mask, -np.inf, scores)` | `attn_mask` 인자로 전달 |
| FeedForward | `Linear → GELU → Linear` 직접 계산 | `nn.Sequential(...)` |
| LayerNorm | `(x - mean) / sqrt(var + eps)` 직접 계산 | `nn.LayerNorm` |
| 잔차 연결 | `x + sublayer(x)` | 코드에서 그대로 `x = x + ...` |

이제 이 대응 관계를 기억하면서 실제 PyTorch 코드를 봅시다. **PyTorch는 우리가 방금 손으로 계산한 것과 완전히 같은 일을, 학습 가능한 가중치와 함께 자동으로 해줍니다.**

---
# Part B. PyTorch로 실제 구현하기

여기서부터는 원본 실습 코드와 동일한 목표(완전한 Transformer Encoder 구현)를 실제 PyTorch `nn.Module`로 작성합니다. 각 클래스 옆에는 Part A의 어떤 계산에 대응하는지 주석으로 표시해두었습니다.

### 실행 환경 안내
아래 코드 셀들은 **PyTorch(torch)** 가 설치되어 있어야 실행됩니다.

- Google Colab을 사용 중이라면 별도 설치 없이 바로 실행됩니다.
- 로컬/서버 환경이라면 아래 명령어로 먼저 설치하세요.

```bash
pip install torch
```

- 이 실습 환경 자체에는 PyTorch가 설치되어 있지 않아(용량이 매우 크고, 이 환경은 GPU 없이 CPU만 사용합니다) 아래 코드 셀들의 출력은 비어 있습니다. Colab 등에 코드를 그대로 복사해서 실행해보시면, Part A에서 손으로 계산했던 것과 동일한 원리가 훨씬 짧은 코드로 자동화되어 있는 것을 확인할 수 있습니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# torch                        : 텐서(다차원 배열) 연산과 자동 미분(autograd)을 제공하는 핵심 라이브러리
# torch.nn                     : Linear, Embedding, LayerNorm 같은 '학습 가능한 레이어'들을 담고 있는 모듈
# torch.nn.functional (F)      : softmax, gelu처럼 '학습되는 가중치가 없는' 함수형 연산 모음

# 실행할 때마다 같은 초기 가중치가 나오도록 시드를 고정합니다 (재현성을 위해).
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 8. FeedForward 클래스 구현

Part A-6에서 손으로 계산했던 FeedForward를 PyTorch `nn.Module`로 그대로 옮깁니다. `nn.Sequential`은 레이어들을 순서대로 이어붙여 하나의 모듈처럼 쓸 수 있게 해주는 컨테이너입니다.

In [ ]:
class FeedForward(nn.Module):
    """
    위치별 완전연결 신경망(Position-wise Feed-Forward Network).
    Part A-6의 W1, b1, gelu(), W2, b2 계산을 그대로 옮긴 것입니다.
    구조: d_model -> (확장) -> d_ff -> (활성화) -> (축소) -> d_model
    """

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 확장: Part A의 (x @ W1 + b1) 에 해당
            nn.GELU(),                  # 활성화 함수: Part A의 gelu() 함수에 해당
            nn.Dropout(dropout),        # 학습 중 일부 값을 랜덤하게 0으로 만들어 과적합(overfitting)을 방지
            nn.Linear(d_ff, d_model),   # 축소: Part A의 (hidden @ W2 + b2) 에 해당
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x shape: (batch, seq_len, d_model) -> 출력도 동일한 shape (다음 층에 그대로 넘길 수 있어야 하기 때문)
        return self.net(x)


# 작은 예시로 shape이 그대로 유지되는지 확인해봅니다. (Colab 등에서 실행해보세요)
ff_demo = FeedForward(d_model=8, d_ff=32)
x_demo = torch.randn(1, 3, 8)          # (batch=1, seq_len=3, d_model=8)
y_demo = ff_demo(x_demo)
print("FeedForward 입력 shape :", x_demo.shape)
print("FeedForward 출력 shape :", y_demo.shape, " (입력과 동일해야 정상)")

## 9. EncoderBlock 클래스 구현

앞서 로드맵에서 본 EncoderBlock 하나를 그대로 코드로 옮깁니다. 구조는 다음과 같습니다 (Pre-LN 방식).

```
x --------------------------------┐
│                                  │
▼                                  │
LayerNorm                         │
▼                                  │
Self-Attention                    │
▼                                  │
Dropout                           │
▼                                  │
+ <--------------------------------┘   (잔차 연결)
│
├----------------------------------┐
▼                                  │
LayerNorm                         │
▼                                  │
FeedForward                       │
▼                                  │
Dropout                           │
▼                                  │
+ <--------------------------------┘   (잔차 연결)
▼
출력
```

'Pre-LN(Pre-LayerNorm)'이라는 이름은 서브층(Attention, FFN)에 들어가기 **전에** LayerNorm을 적용한다는 뜻입니다. 원 논문은 반대로 서브층을 통과하고 잔차를 더한 **후에** 적용하는 Post-LN을 사용했지만, Pre-LN이 층을 깊게 쌓았을 때 학습이 더 안정적이라 최근 구현체들은 대부분 Pre-LN을 사용합니다.

In [ ]:
class EncoderBlock(nn.Module):
    """
    Transformer Encoder의 한 개 블록(층).
    위 다이어그램을 그대로 코드로 옮긴 것입니다.
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # PyTorch가 제공하는 완성된 Multi-Head Attention 모듈입니다.
        # Part A에서 손으로 계산했던 'Q@K.T -> scale -> softmax -> @V'와 '여러 head로 나누기' 과정 전체가
        # 이 한 줄 안에 구현되어 있고, W_Q/W_K/W_V/W_O에 해당하는 가중치도 내부에서 자동으로 학습됩니다.
        # batch_first=True : 입력 shape을 (seq_len, batch, d_model)이 아니라
        #                     우리에게 익숙한 (batch, seq_len, d_model) 순서로 쓰겠다는 옵션입니다.
        self.self_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)  # Self-Attention 앞에 붙는 정규화
        self.norm2 = nn.LayerNorm(d_model)  # FeedForward 앞에 붙는 정규화
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # x shape: (batch, seq_len, d_model) - 이 shape은 블록을 통과해도 절대 바뀌지 않습니다!
        # (그래야 다음 EncoderBlock에 그대로 넘겨서 여러 겹 쌓을 수 있습니다)

        # --- 서브층 1: Self-Attention ---
        normed = self.norm1(x)
        # Self-Attention이므로 Query, Key, Value 자리에 모두 같은 값(normed)을 넣습니다.
        # (자기 자신을 기준으로, 자기 자신들끼리의 관계를 계산하기 때문에 'Self'-Attention 입니다)
        attn_out, _ = self.self_attn(normed, normed, normed, attn_mask=src_mask)
        x = x + self.dropout(attn_out)  # 잔차 연결 (Part A: residual_output = attn_output + ... 과 같은 아이디어)

        # --- 서브층 2: FeedForward ---
        x = x + self.dropout(self.ffn(self.norm2(x)))

        return x

## 10. TransformerEncoder 전체 모델

`EncoderBlock`을 `num_layers`개만큼 쌓고, 앞뒤로 Embedding과 마지막 LayerNorm을 붙이면 전체 모델이 완성됩니다.

In [ ]:
class TransformerEncoder(nn.Module):
    """
    EncoderBlock을 num_layers개 쌓아서 만든 전체 Encoder 모델입니다.
    """

    def __init__(self, vocab_size, d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, max_seq_len=512):
        super().__init__()
        # 1) 토큰 ID -> 의미 벡터 (Part A의 embedding_table[token_ids] 에 해당)
        self.embedding = nn.Embedding(vocab_size, d_model)
        # 2) 위치 ID -> 위치 벡터 (Part A의 position_table[positions] 에 해당)
        self.pos_encoding = nn.Embedding(max_seq_len, d_model)
        # 3) EncoderBlock을 num_layers개 쌓습니다.
        #    nn.ModuleList를 사용해야 내부 각 층의 파라미터를 PyTorch가 제대로 인식하고 학습합니다.
        #    (일반 파이썬 list를 쓰면 파라미터가 인식되지 않아 학습이 안 되는 흔한 실수이니 주의하세요!)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        # 4) 마지막에 한 번 더 정규화 (Pre-LN 구조에서는 관례적으로 맨 끝에 LayerNorm을 하나 더 둡니다)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x shape: (batch, seq_len) - 아직은 정수 토큰 ID 상태입니다. 예: [[15, 302, 9981, ...], ...]
        seq_len = x.size(1)

        # 0, 1, 2, ..., seq_len-1 위치 인덱스를 만듭니다.
        positions = torch.arange(seq_len, device=x.device)

        # 토큰 임베딩 + 위치 임베딩 (Part A-2에서 x = token_emb + pos_emb 했던 것과 동일)
        # embedding(x)            : (batch, seq_len, d_model)
        # pos_encoding(positions) : (seq_len, d_model) -> 브로드캐스팅으로 batch 차원에 자동으로 맞춰짐
        x = self.embedding(x) + self.pos_encoding(positions)

        # 쌓아놓은 EncoderBlock들을 순서대로 통과시킵니다.
        for layer in self.layers:
            x = layer(x)

        return self.norm(x)

## 11. 실제로 테스트해보기

이제 실제 크기(원본 실습 코드와 동일한 하이퍼파라미터)로 모델을 만들고, 무작위 토큰 ID를 입력으로 넣어 shape과 파라미터 개수를 확인합니다.

In [ ]:
torch.manual_seed(42)

model = TransformerEncoder(
    vocab_size=30000,   # 사전에 등록된 단어(토큰) 개수
    d_model=256,         # 각 토큰을 표현하는 벡터의 차원
    num_heads=4,         # Multi-Head Attention의 head 개수  (256 / 4 = 64 -> 나누어 떨어져야 함!)
    d_ff=1024,           # FeedForward 내부의 확장 차원 (보통 d_model의 4배)
    num_layers=4,        # EncoderBlock을 몇 겹 쌓을지
)

# 실제 문장 대신, 0~29999 사이의 랜덤한 정수로 이루어진 '가짜 토큰 ID'를 만듭니다.
# shape: (batch_size=2, seq_len=64) -> 문장 2개, 각 문장은 토큰 64개로 구성
x = torch.randint(0, 30000, (2, 64))

out = model(x)

print(f"입력 shape : {x.shape}   (batch_size=2, seq_len=64)")
print(f"출력 shape : {out.shape}   (batch_size=2, seq_len=64, d_model=256)")
print("  -> seq_len(64)은 그대로, 마지막 차원만 d_model(256)로 바뀐 것을 확인하세요.")
print(f"전체 파라미터 개수: {sum(p.numel() for p in model.parameters()):,}")

## 12. (보너스) 직접 실험해보기

원본 실습 코드의 "주의" 사항에 있던 제안대로, 하이퍼파라미터를 직접 바꿔보며 결과가 어떻게 달라지는지 실험해봅니다.

In [ ]:
# 1) d_model이 num_heads로 나누어떨어지지 않으면 어떤 에러가 날까요?
#    (Part A-4에서 손으로 확인했던 assert와 같은 원리로, PyTorch 내부에서도 동일하게 검사합니다)
try:
    broken_model = TransformerEncoder(vocab_size=1000, d_model=256, num_heads=5, d_ff=512, num_layers=1)
except Exception as e:
    print("[예상되는 에러]", type(e).__name__, "-", e)

# 2) 하이퍼파라미터를 바꿔가며 파라미터 개수가 어떻게 달라지는지 비교해봅시다.
configs = [
    dict(d_model=128, num_heads=4, d_ff=512,  num_layers=2),
    dict(d_model=256, num_heads=4, d_ff=1024, num_layers=4),
    dict(d_model=512, num_heads=8, d_ff=2048, num_layers=6),
]

for cfg in configs:
    m = TransformerEncoder(vocab_size=30000, **cfg)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{cfg}  ->  파라미터 {n_params:,}개")

# 3) 이 밖에 바꿔볼 수 있는 것들
#    - torch.manual_seed(...) 값을 바꾸면 (구조는 그대로, 초기 가중치 값만) 달라지는지 확인해보세요.
#    - max_seq_len보다 긴 seq_len을 입력으로 넣으면 어떤 에러가 나는지 확인해보세요.
#    - EncoderBlock.forward()에 src_mask를 실제로 전달해서 Part A-5의 causal mask를 적용해보세요.

## 정리

이번 실습에서 다룬 내용을 처음의 학습 목표에 맞춰 정리하면 다음과 같습니다.

1. **Q/K/V shape 추적** — Part A-3에서 `Q = x @ W_q` 등으로 직접 Q, K, V를 만들고, `Q @ K.T` → 스케일링 → `softmax` → `@ V`를 거치며 shape이 `(seq_len, d_model)`로 유지되는 과정을 직접 확인했습니다.
2. **마스킹 후 softmax가 0이 되는 원리** — Part A-5에서 `-inf`로 채운 위치가 softmax를 거치며 정확히 `0.000`이 되는 것을 직접 눈으로 확인했습니다.

그리고 Part B에서는 이 모든 원리가 PyTorch의 `nn.Embedding`, `nn.MultiheadAttention`, `nn.LayerNorm`, `nn.Sequential` 등으로 이미 구현되어 있어서, 우리는 이 부품들을 순서대로 조립하기만 하면 된다는 것도 확인했습니다.

### 다음에 시도해보면 좋은 것들
- Encoder(양방향)가 아니라 Decoder(단방향, causal mask 적용)를 직접 만들어보기
- `nn.MultiheadAttention` 대신 Part A처럼 Q/K/V를 직접 `nn.Linear`로 만들어서 Multi-Head Attention을 밑바닥부터 구현해보기
- 실제 텍스트 데이터를 토크나이저(예: Hugging Face `tokenizers`, `tiktoken`)로 토큰화해서 진짜 문장을 넣어 결과를 확인해보기